---
## Cell 1 — Install dependencies

In [ ]:
!pip install ultralytics roboflow albumentations PyYAML -q

import torch
import os
import shutil
import yaml
from pathlib import Path
from collections import Counter

print('=== Environment Check ===')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

---
## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#output folders
DRIVE_OUTPUT = '/content/drive/MyDrive/ow_tracker'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/weights', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/exports', exist_ok=True)

print(f'Drive mounted. Outputs will save to: {DRIVE_OUTPUT}')

In [ ]:
#Import Datasets from Roboflow
from roboflow import Roboflow

API_KEY = os.getenv('API_KEY')

rf = Roboflow(api_key=API_KEY)

print('Downloading dataset 1/5...')
proj1 = rf.workspace('atharvas-workspace-bu7ge').project('overwatch-ejbtc-fcpfq')
ds1 = proj1.version(1).download('yolov8')

print('Downloading dataset 2/5...')
proj2 = rf.workspace('atharvas-workspace-bu7ge').project('overwatch-mdqg0-ij1vi')
ds2 = proj2.version(1).download('yolov8')

print('Downloading dataset 3/5...')
proj3 = rf.workspace('atharvas-workspace-bu7ge').project('overwatch-mdqg0-uetwl')
ds3 = proj3.version(1).download('yolov8')

print('Downloading dataset 4/5...')
proj4 = rf.workspace('atharvas-workspace-bu7ge').project('overwatch-llcjh-1atk')
ds4 = proj4.version(1).download('yolov8')

print('Downloading dataset 5/5...')
proj5 = rf.workspace('atharvas-workspace-bu7ge').project('overwatch-bot-tar5g')
ds5 = proj5.version(1).download('yolov8')

# Print where each dataset landed
for i, ds in enumerate([ds1, ds2, ds3, ds4, ds5], 1):
    print(f'Dataset {i}: {ds.location}')

In [ ]:
#Get class names from each dataset and print them
def get_classes(dataset):
    yaml_path = Path(dataset.location) / 'data.yaml'
    with open(yaml_path) as f:
        data = yaml.safe_load(f)
    return data.get('names', [])

datasets = [ds1, ds2, ds3, ds4, ds5]
all_classes_per_ds = []

for i, ds in enumerate(datasets, 1):
    classes = get_classes(ds)
    all_classes_per_ds.append(classes)
    print(f'\nDataset {i} ({len(classes)} classes):')
    for j, c in enumerate(classes):
        print(f'  [{j}] {c}')

# Show all unique classes across every dataset
all_unique = sorted(set(c for classes in all_classes_per_ds for c in classes))
print(f'\nTotal unique class names across all datasets: {len(all_unique)}')
for c in all_unique:
    print(f'  {c}')

In [ ]:
#Merge class names for easier mapping later

UNIFIED_CLASSES = [
    'enemy',               # 0  any visible enemy
    'enemy-head',          # 1  enemy head hitbox (from dataset with 3-Enemy-Head)
    'enemy-behind-object', # 2
    'enemy-shield',        # 3
    'enemy-trap',          # 4
    'friendly',            # 5  any visible teammate
    'friendly-behind-object', # 6
    'friendly-low-health', # 7
    'dead-friendly',       # 8
    'respawning',          # 9
    'opponent-player',     # 10 (from Opponent-Player)
    'team-player',         # 11 (from TeamPlayer)

    'health',              # 12 health bar
    'health-pack',         # 13 generic health pack
    'health-pack-ready',   # 14
    'health-pack-not-ready', # 15
    'ult-ready',           # 16
    'ult-not-ready',       # 17 (ult-notready → ult-not-ready)
    'ultimate',            # 18 ultimate charge bar area
    'orb',                 # 19
    'orb-ready',           # 20
    'orb-not-ready',       # 21
    'fade',                # 22 (Moira fade ability)
    'fade-ready',          # 23
    'fade-not-ready',      # 24
    'piss',                # 25 (Roadhog / Junker Queen ability indicator)
    'piss-full',           # 26
    'piss-low',            # 27
    'score',               # 28
    'objective',           # 29
    'obj-locked',          # 30
    'obj-unlocked',        # 31
    'ui',                  # 32 generic UI (from UserInterface)
    'in-game-object',      # 33 (from InGameObject)

    'kill',                # 34
    'death',               # 35
    'assist',              # 36
    'onfire',              # 37
]

CLASS_TO_IDX = {name: i for i, name in enumerate(UNIFIED_CLASSES)}

#remaps from original dataset class names to the unified class names 
REMAP = {
    'Enemy':                  'enemy',
    'Friendly':               'friendly',
    '1-Friendly-Body':        'friendly',
    '2-Enemy-Body':           'enemy',
    '3-Enemy-Head':           'enemy-head',
    'Opponent-Player':        'opponent-player',
    'TeamPlayer':             'team-player',
    'UserInterface':          'ui',
    'InGameObject':           'in-game-object',
    'ult-notready':           'ult-not-ready',
    'ult-ready':              'ult-ready',
    'dead-friendly':          'dead-friendly',
    'enemy-behind-object':    'enemy-behind-object',
    'enemy-shield':           'enemy-shield',
    'enemy-trap':             'enemy-trap',
    'fade':                   'fade',
    'fade-not-ready':         'fade-not-ready',
    'fade-ready':             'fade-ready',
    'friendly-behind-object': 'friendly-behind-object',
    'friendly-low-health':    'friendly-low-health',
    'health':                 'health',
    'health-pack':            'health-pack',
    'health-pack-not-ready':  'health-pack-not-ready',
    'health-pack-ready':      'health-pack-ready',
    'obj-locked':             'obj-locked',
    'obj-unlocked':           'obj-unlocked',
    'objective':              'objective',
    'onfire':                 'onfire',
    'orb':                    'orb',
    'orb-not-ready':          'orb-not-ready',
    'orb-ready':              'orb-ready',
    'piss':                   'piss',
    'piss-full':              'piss-full',
    'piss-low':               'piss-low',
    'respawning':             'respawning',
    'score':                  'score',
    'ultimate':               'ultimate',
    'assist':                 'assist',
    'death':                  'death',
    'kill':                   'kill',
}

print(f'Unified class count: {len(UNIFIED_CLASSES)}')
print('Classes:')
for i, c in enumerate(UNIFIED_CLASSES):
    print(f'  [{i:2d}] {c}')

In [ ]:
#Merge datasets into one folder
MERGED_DIR = Path('/content/ow_merged')
for split in ['train', 'valid', 'test']:
    (MERGED_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

skipped = 0
copied = 0
class_counter = Counter()

def remap_label_file(src_label: Path, dst_label: Path, original_classes: list):
    """Read a YOLO label file, remap class indices, write to dst."""
    global skipped, copied
    lines_out = []
    if not src_label.exists():
        return False
    with open(src_label) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            old_idx = int(parts[0])
            if old_idx >= len(original_classes):
                continue
            original_name = original_classes[old_idx]
            unified_name = REMAP.get(original_name)
            if unified_name is None:
                #skip classes that i dont need
                skipped += 1
                continue
            new_idx = CLASS_TO_IDX.get(unified_name)
            if new_idx is None:
                skipped += 1
                continue
            class_counter[unified_name] += 1
            lines_out.append(f'{new_idx} {" ".join(parts[1:])}\n')
    if lines_out:
        with open(dst_label, 'w') as f:
            f.writelines(lines_out)
        copied += 1
        return True
    return False

def merge_dataset(dataset, ds_idx):
    ds_path = Path(dataset.location)
    original_classes = all_classes_per_ds[ds_idx]
    print(f'\nMerging dataset {ds_idx+1} from {ds_path}')

    for split in ['train', 'valid', 'test']:
        img_dir = ds_path / split / 'images'
        lbl_dir = ds_path / split / 'labels'
        if not img_dir.exists():
            print(f'  No {split} split found, skipping.')
            continue
        images = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
        n = 0
        for img_path in images:
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            new_stem = f'ds{ds_idx+1}_{img_path.stem}'
            dst_img = MERGED_DIR / split / 'images' / (new_stem + img_path.suffix)
            dst_lbl = MERGED_DIR / split / 'labels' / (new_stem + '.txt')
            shutil.copy2(img_path, dst_img)
            remap_label_file(lbl_path, dst_lbl, original_classes)
            n += 1
        print(f'  {split}: {n} images')

for i, ds in enumerate(datasets):
    merge_dataset(ds, i)

# print final merged dataset summary
for split in ['train', 'valid', 'test']:
    n_img = len(list((MERGED_DIR / split / 'images').glob('*')))
    n_lbl = len(list((MERGED_DIR / split / 'labels').glob('*.txt')))
    print(f'  {split}: {n_img} images, {n_lbl} label files')

print(f'\nAnnotations copied : {copied}')
print(f'Annotations skipped (unmapped): {skipped}')
print('\nClass distribution in training set:')
for cls, count in class_counter.most_common():
    bar = '█' * min(40, count // max(1, max(class_counter.values()) // 40))
    print(f'  {cls:<28} {count:5d}  {bar}')

In [ ]:
#make merged yaml file for training
data_yaml = {
    'path': str(MERGED_DIR),
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    len(UNIFIED_CLASSES),
    'names': UNIFIED_CLASSES,
}

yaml_path = MERGED_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f'data.yaml written to {yaml_path}')
print(f'Number of classes: {len(UNIFIED_CLASSES)}')

#print yaml file
with open(yaml_path) as f:
    print(f.read())

In [ ]:
#add class weights to yaml

import numpy as np
from pathlib import Path

def count_class_instances(labels_dir: Path, n_classes: int):
    counts = np.zeros(n_classes)
    for lbl_file in labels_dir.glob('*.txt'):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counts[int(parts[0])] += 1
    return counts

train_lbl_dir = MERGED_DIR / 'train' / 'labels'
counts = count_class_instances(train_lbl_dir, len(UNIFIED_CLASSES))

print('Class counts in training set:')
zero_classes = []
for i, (cls, count) in enumerate(zip(UNIFIED_CLASSES, counts)):
    status = 'no samples' if count == 0 else ''
    print(f'  [{i:2d}] {cls:<28} {int(count):5d}{status}')
    if count == 0:
        zero_classes.append(cls)

if zero_classes:
    print(f'\nWARNING: {len(zero_classes)} classes have zero training samples:')
    for c in zero_classes:
        print(f'  - {c}')

In [ ]:
#Run yolov8m model with ultralytics using coco pretrained weights and dataset
from ultralytics import YOLO
import shutil

model = YOLO('yolov8m.pt')  # downloads pretrained COCO weights

results = model.train(
    data=str(MERGED_DIR / 'data.yaml'),
    imgsz=640,

    epochs=100,
    patience=25,         # early stop if no improvement for 25 epochs
    batch=16,            # reduce to 8 if you get CUDA out-of-memory

    optimizer='AdamW',
    lr0=0.001,           # initial learning rate
    lrf=0.01,            # final lr = lr0 * lrf
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,     # gradual warmup prevents early instability
    cos_lr=True,         # cosine annealing schedule

    dropout=0.1,
    label_smoothing=0.1, # prevents overconfidence on noisy game labels

    #augmentation
    hsv_h=0.015,         # hue shift 
    hsv_s=0.7,           # saturation 
    hsv_v=0.4,           # brightness 
    degrees=5.0,         # slight rotation
    translate=0.1,
    scale=0.5,           # zoom
    shear=2.0,
    perspective=0.0,
    flipud=0.0,          #  gameplay is always upright
    fliplr=0.5,          # horizontal flip is safe
    mosaic=1.0,          # biggest single accuracy boost — composites 4 images
    mixup=0.15,          # blends two training images
    copy_paste=0.1,      # pastes objects from other images
    erasing=0.4,         # helps with partially hidden players

    box=7.5,             # bounding box regression loss
    cls=0.5,             # classification loss
    dfl=1.5,             # distribution focal loss

    project='/content/ow_tracker',
    name='run1',
    exist_ok=True,
    save=True,
    save_period=10,      # checkpoint every 10 epochs
    val=True,
    plots=True,          # saves training curves, confusion matrix
    verbose=True,
)

# Immediately back up best weights to Drive
best_pt = '/content/ow_tracker/run1/weights/best.pt'
shutil.copy(best_pt, f'{DRIVE_OUTPUT}/weights/run1_best.pt')
print(f'Best weights backed up to Drive.')

In [ ]:
#print training results and metrics

from ultralytics import YOLO
import json

model = YOLO('/content/ow_tracker/run1/weights/best.pt')

# Evaluate on test split
metrics = model.val(
    data=str(MERGED_DIR / 'data.yaml'),
    split='test',
    conf=0.001,   # low threshold to measure full precision-recall curve
    iou=0.6,
    plots=True,
)

print('\n Metrics')
print(f'mAP@50      : {metrics.box.map50:.4f}   (target: >0.70)')
print(f'mAP@50-95   : {metrics.box.map:.4f}   (stricter, usually lower)')
print(f'Precision   : {metrics.box.mp:.4f}')
print(f'Recall      : {metrics.box.mr:.4f}')

print('\n mAP50 per class:')
if hasattr(metrics.box, 'ap_class_index'):
    for idx, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
        cls_name = UNIFIED_CLASSES[idx] if idx < len(UNIFIED_CLASSES) else str(idx)
        bar = '█' * int(ap * 30)
        status = ' needs work' if ap < 0.5 else ''
        print(f'  {cls_name:<28} {ap:.3f}  {bar}{status}')

In [ ]:
#check results visually on random test images

import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from ultralytics import YOLO

model = YOLO('/content/ow_tracker/run1/weights/best.pt')

# Pick 6 random images from the test set
test_images = list((MERGED_DIR / 'test' / 'images').glob('*.jpg'))
test_images += list((MERGED_DIR / 'test' / 'images').glob('*.png'))
sample = random.sample(test_images, min(6, len(test_images)))

results = model.predict(
    source=sample,
    conf=0.35,
    iou=0.5,
    save=True,
    project='/content/ow_tracker',
    name='visual_check',
    exist_ok=True,
)

# Display in notebook
pred_dir = Path('/content/ow_tracker/visual_check')
pred_images = list(pred_dir.glob('*.jpg')) + list(pred_dir.glob('*.png'))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img_path in zip(axes.flatten(), pred_images[:6]):
    ax.imshow(mpimg.imread(img_path))
    ax.axis('off')
    ax.set_title(img_path.name, fontsize=8)
plt.tight_layout()
plt.savefig(f'{DRIVE_OUTPUT}/sample_predictions.png', dpi=150)
plt.show()
print('Visual check saved to Drive.')

In [ ]:
#fine tune the model with a lower learning rate and more epochs

from ultralytics import YOLO
import shutil

# Start from the best checkpoint of run1
model = YOLO('/content/ow_tracker/run1/weights/best.pt')

results = model.train(
    data=str(MERGED_DIR / 'data.yaml'),
    imgsz=640,

    epochs=50,
    patience=15,
    batch=8,             # smaller batch for fine-tuning stability

    optimizer='AdamW',
    lr0=0.0001,          # 10x lower than initial training
    lrf=0.001,
    warmup_epochs=1,
    cos_lr=True,

    #this makes it so that only the detection layers are retrained
    freeze=10,

    hsv_h=0.01,
    hsv_s=0.4,
    hsv_v=0.3,
    fliplr=0.5,
    mosaic=0.5,         
    mixup=0.05,
    copy_paste=0.05,

    label_smoothing=0.05,
    dropout=0.05,

    project='/content/ow_tracker',
    name='run2_finetune',
    exist_ok=True,
    save=True,
    save_period=5,
    val=True,
    plots=True,
)

# Back up fine-tuned weights
best_ft = '/content/ow_tracker/run2_finetune/weights/best.pt'
shutil.copy(best_ft, f'{DRIVE_OUTPUT}/weights/run2_finetune_best.pt')

# Compare run1 vs run2
print('\nFine-tune complete. Run Cell 10 again pointing to run2_finetune/weights/best.pt to compare.')

In [ ]:
#find best inference confidence

from ultralytics import YOLO
import matplotlib.pyplot as plt

#use whichever weights are better
BEST_WEIGHTS = '/content/ow_tracker/run2_finetune/weights/best.pt'
model = YOLO(BEST_WEIGHTS)

thresholds = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
results_by_conf = []

for conf in thresholds:
    m = model.val(
        data=str(MERGED_DIR / 'data.yaml'),
        split='test',
        conf=conf,
        iou=0.5,
        verbose=False,
    )
    results_by_conf.append({
        'conf': conf,
        'mAP50': m.box.map50,
        'precision': m.box.mp,
        'recall': m.box.mr,
    })
    print(f'conf={conf:.2f}  mAP50={m.box.map50:.4f}  P={m.box.mp:.3f}  R={m.box.mr:.3f}')

best = max(results_by_conf, key=lambda x: x['mAP50'])
print(f'\nBest confidence threshold: {best["conf"]} (mAP50={best["mAP50"]:.4f})')
print(f'Use conf={best["conf"]} in your FastAPI inference code.')

# Plot
confs = [r['conf'] for r in results_by_conf]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(confs, [r['mAP50'] for r in results_by_conf], 'o-', label='mAP50')
ax.plot(confs, [r['precision'] for r in results_by_conf], 's-', label='Precision')
ax.plot(confs, [r['recall'] for r in results_by_conf], '^-', label='Recall')
ax.axvline(best['conf'], color='red', linestyle='--', alpha=0.5, label=f'Best: {best["conf"]}')
ax.set_xlabel('Confidence threshold')
ax.set_ylabel('Score')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_OUTPUT}/confidence_sweep.png', dpi=150)
plt.show()

In [ ]:
#export weights to onnx and pytorch for backend

from ultralytics import YOLO
import shutil

BEST_WEIGHTS = '/content/ow_tracker/run2_finetune/weights/best.pt'
model = YOLO(BEST_WEIGHTS)

# Export to ONNX
export_path = model.export(
    format='onnx',
    imgsz=640,
    optimize=True,     # optimize for inference speed
    simplify=True,     # simplify ONNX graph
    dynamic=False,     # fixed batch size = faster
    half=False,        # keep float32 for CPU compatibility
)

print(f'Exported to: {export_path}')

# Save everything to Drive
shutil.copy(BEST_WEIGHTS, f'{DRIVE_OUTPUT}/exports/best_final.pt')
shutil.copy(str(export_path), f'{DRIVE_OUTPUT}/exports/best_final.onnx')
shutil.copy(str(MERGED_DIR / 'data.yaml'), f'{DRIVE_OUTPUT}/exports/data.yaml')

# Write class names file for the backend
with open(f'{DRIVE_OUTPUT}/exports/classes.txt', 'w') as f:
    for cls in UNIFIED_CLASSES:
        f.write(cls + '\n')

import os
pt_size   = os.path.getsize(f'{DRIVE_OUTPUT}/exports/best_final.pt')   / 1e6
onnx_size = os.path.getsize(f'{DRIVE_OUTPUT}/exports/best_final.onnx') / 1e6
print(f'\nFiles saved to {DRIVE_OUTPUT}/exports/')
print(f'  best_final.pt   : {pt_size:.1f} MB  (use for further training)')
print(f'  best_final.onnx : {onnx_size:.1f} MB  (use in FastAPI backend)')
print(f'  data.yaml       : class definitions')
print(f'  classes.txt     : class names, one per line')

print('\nTo use in FastAPI:')
print('  1. Download best_final.onnx and classes.txt from Drive')
print('  2. Place in backend/models/')
print('  3. Load with: model = YOLO("models/best_final.onnx")')

In [ ]:
#check how fast the model runs on CPU vs GPU

from ultralytics import YOLO
import time
import numpy as np

# Test both .pt and .onnx
for model_path, label in [
    (f'{DRIVE_OUTPUT}/exports/best_final.pt',   'PyTorch (.pt) '),
    (f'{DRIVE_OUTPUT}/exports/best_final.onnx', 'ONNX          '),
]:
    model = YOLO(model_path)
    # Warmup
    dummy = np.zeros((640, 640, 3), dtype=np.uint8)
    for _ in range(3):
        model(dummy, verbose=False)

    # Benchmark
    times = []
    for _ in range(20):
        t0 = time.perf_counter()
        model(dummy, verbose=False)
        times.append(time.perf_counter() - t0)

    avg_ms = np.mean(times) * 1000
    fps = 1000 / avg_ms
    print(f'{label}  avg: {avg_ms:.1f}ms/frame  ({fps:.1f} FPS theoretical max)')

print('\nRecommended backend frame sampling:')
print('  GPU server  : every frame or every 2nd frame')
print('  CPU server  : every 5th–10th frame (set stride=5 in tracker.py)')